# Predikcija kategorije proizvoda na osnovu naslova

**Autor:** Stefan Stojanović

## Cilj projekta

Cilj ovog projekta je razvoj modela mašinskog učenja koji na osnovu naziva proizvoda automatski predlaže odgovarajuću kategoriju.

Projekat obuhvata analizu i čišćenje podataka, inženjering karakteristika, treniranje i poređenje više modela, evaluaciju najboljeg rešenja i njegovo čuvanje za kasniju upotrebu.

Razvijeni model može doprineti bržem unosu proizvoda, smanjenju broja grešaka pri ručnoj kategorizaciji i jednostavnijem radu zaposlenih na platformi za online trgovinu.

## 1. Uvoz potrebnih biblioteka

U ovom delu uvozimo biblioteke potrebne za rad sa podacima, vizualizaciju rezultata i razvoj modela mašinskog učenja.

In [1]:
# Biblioteke za učitavanje i obradu podataka
import pandas as pd
import numpy as np

# Biblioteke za vizualizaciju
import matplotlib.pyplot as plt
import seaborn as sns

# Pomoćne biblioteke
import re
import time
from pathlib import Path

# Podešavanje prikaza tabela
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

# Podešavanje izgleda grafikona
sns.set_theme(style="whitegrid")

print("Sve biblioteke su uspešno učitane.")

Sve biblioteke su uspešno učitane.


## 2. Učitavanje i početni pregled podataka

Učitavamo skup podataka iz foldera `data` i proveravamo njegove dimenzije i prve redove. Putanja je podešena tako da sveska može da radi i kada se pokrene iz glavnog foldera projekta i kada se pokrene iz foldera `notebooks`.

In [2]:
# Moguće putanje do skupa podataka
possible_paths = [
    Path("../data/products.csv"),
    Path("data/products.csv")
]

# Pronalazimo prvu postojeću putanju
data_path = next((path for path in possible_paths if path.exists()), None)

# Zaustavljamo izvršavanje uz jasnu poruku ako fajl nije pronađen
if data_path is None:
    raise FileNotFoundError(
        "Fajl products.csv nije pronađen u folderu data."
    )

# Učitavanje podataka
df = pd.read_csv(data_path)

print(f"Dataset je uspešno učitan sa putanje: {data_path}")
print(f"Broj redova: {df.shape[0]}")
print(f"Broj kolona: {df.shape[1]}")

# Prikaz prvih pet redova
df.head()

Dataset je uspešno učitan sa putanje: ..\data\products.csv
Broj redova: 35311
Broj kolona: 8


,product ID,Product Title,Merchant ID,Category Label,_Product Code,Number_of_Views,Merchant Rating,Listing Date
0,1,apple iphone 8 plus 64gb silver,1,Mobile Phones,QA-2276-XC,860.0,2.5,5/10/2024
1,2,apple iphone 8 plus 64 gb spacegrau,2,Mobile Phones,KA-2501-QO,3772.0,4.8,12/31/2024
2,3,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim free smartphone in gold,3,Mobile Phones,FP-8086-IE,3092.0,3.9,11/10/2024
3,4,apple iphone 8 plus 64gb space grey,4,Mobile Phones,YI-0086-US,466.0,3.4,5/2/2022
4,5,apple iphone 8 plus gold 5.5 64gb 4g unlocked sim free,5,Mobile Phones,NZ-3586-WP,4426.0,1.6,4/12/2023


### 2.1. Struktura skupa i tipovi podataka

Proveravamo tačne nazive kolona, tipove podataka i broj popunjenih vrednosti. Ovaj pregled pomaže da otkrijemo skrivene razmake u nazivima kolona i kolone koje sadrže nedostajuće podatke.

In [3]:
# Prikaz tačnih naziva kolona
print("Nazivi kolona:")
for column in df.columns:
    print(repr(column))

print("\nOsnovne informacije o skupu podataka:")
df.info()

Nazivi kolona:
'product ID'
'Product Title'
'Merchant ID'
' Category Label'
'_Product Code'
'Number_of_Views'
'Merchant Rating'
' Listing Date  '

Osnovne informacije o skupu podataka:
<class 'pandas.DataFrame'>
RangeIndex: 35311 entries, 0 to 35310
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   product ID       35311 non-null  int64  
 1   Product Title    35139 non-null  str    
 2   Merchant ID      35311 non-null  int64  
 3    Category Label  35267 non-null  str    
 4   _Product Code    35216 non-null  str    
 5   Number_of_Views  35297 non-null  float64
 6   Merchant Rating  35141 non-null  float64
 7    Listing Date    35252 non-null  str    
dtypes: float64(2), int64(2), str(4)
memory usage: 2.2 MB


### 2.2. Nedostajuće vrednosti i duplikati

Proveravamo broj i procenat nedostajućih vrednosti u svakoj koloni, kao i postojanje potpuno dupliranih redova. Ovi rezultati će odrediti naredne korake čišćenja podataka.

In [4]:
# Broj nedostajućih vrednosti po kolonama
missing_values = df.isna().sum()

# Procenat nedostajućih vrednosti po kolonama
missing_percentage = (missing_values / len(df) * 100).round(2)

# Tabela sa rezultatima
missing_summary = pd.DataFrame({
    "Broj nedostajućih": missing_values,
    "Procenat (%)": missing_percentage
})

print("Pregled nedostajućih vrednosti:")
display(missing_summary)

# Provera potpuno dupliranih redova
duplicate_rows = df.duplicated().sum()

print(f"\nBroj potpuno dupliranih redova: {duplicate_rows}")

Pregled nedostajućih vrednosti:


,Broj nedostajućih,Procenat (%)
product ID,0,0.00
Product Title,172,0.49
Merchant ID,0,0.00
Category Label,44,0.12
_Product Code,95,0.27
Number_of_Views,14,0.04
Merchant Rating,170,0.48
Listing Date,59,0.17



Broj potpuno dupliranih redova: 0
